# Appendix — Our Cannon against a published implementation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/the-cannon-tutorial/blob/main/notebooks/A_annieslasso.ipynb)

**Goal.** Check the few dozen lines of The Cannon written in Chapter 2 against a published code: *AnniesLasso*
(the code of *The Cannon 2*, Casey et al. 2016), version 0.2.96 (October 2025), installed with
`pip install annieslasso==0.2.96`; it is imported as `thecannon`. Both codes are trained on the same APOGEE spectra
and labels (Chapter 4) and label the same 1500 test stars.

**You will learn**
1. to separate a *bug* from a *difference in method* when two codes disagree: first run them the same way, where
   they must agree to numerical precision, then the way each is meant to be run;
2. that two published ways to train the model — ours (Chapter 2) and that of AnniesLasso — give scatters
   $s_\lambda$ that differ by a factor of two, but labels that differ by much less than their errors.

**Reading the AnniesLasso code** (`thecannon/model.py`, `thecannon/fitting.py`) shows how it trains a model
without regularisation ($\Lambda = 0$):
- the labels are scaled as ours, by the median and the 2.5–97.5 percentile range (we pass our scale, half that
  range, so that the coefficients $\boldsymbol\theta_\lambda$ of the two codes can be compared one to one; the
  order of the terms in the label vector is the same as ours);
- at each pixel $\boldsymbol\theta_\lambda$ is fitted with the scatter set to **zero** (weights $1/\sigma^2$),
  by an L-BFGS minimisation of $\chi^2$ started from the linear-algebra solution;
- then the scatter is chosen, once, so that the **median** of $r^2/(\sigma^2+s^2)$ over the training stars is 1.

Our `train` (Chapter 2) instead iterates $\boldsymbol\theta_\lambda$ with weights $1/(\sigma^2+s^2)$ and $s_\lambda$
by maximum likelihood. The test step is the same in both: a non-linear least-squares fit with weights
$1/(\sigma^2+s^2)$ and the analytic Jacobian, started at the median labels.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are extracts of APOGEE DR17 and Gaia DR3, published on Zenodo (https://doi.org/10.5281/zenodo.23813051).
# MANIFEST.json lists, for each file, the query that made it and its SHA-256 checksum.
DATA_URL = 'https://zenodo.org/records/23813051/files/{}?download=1'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL.format('MANIFEST.json')).read())


def load(name, cache=Path('data')):
    """Download <name> (.parquet table or .npz arrays) from Zenodo if needed, check its SHA-256, read it.

    A .parquet file is returned as a pandas DataFrame, an .npz file as a dict of numpy arrays.
    """
    ext = '.npz' if name.startswith(('spectra_', 'rvs_')) else '.parquet'
    path = cache / f'{name}{ext}'
    sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    if not path.exists() or sha(path) != MANIFEST[name]['sha256']:   # first use, or a stale/damaged copy
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        part = path.with_suffix('.part')
        urllib.request.urlretrieve(DATA_URL.format(path.name), part)   # download to a temporary file first
        if sha(part) != MANIFEST[name]['sha256']:
            raise IOError(f'{path.name}: checksum does not match the manifest (incomplete download?)')
        part.replace(path)                                         # only a verified file takes the final name
    # the checksum guarantees that everyone runs this chapter on exactly the same data
    if ext == '.parquet':
        return pd.read_parquet(path)
    with np.load(path, allow_pickle=False) as f:   # allow_pickle=False: arrays only, no code can be loaded
        return {k: f[k] for k in f.files}
# --- Spectra and continuum normalisation (as in Chapters 0 and 3) ----------------------------------------
NPIX = 8575
wave = 10**(4.179 + 6e-6 * np.arange(NPIX))               # apStar vacuum wavelength grid [Angstrom]
CHIPS = [(15150, 15800), (15865, 16425), (16480, 16950)]  # the three detectors [A], inside the edges of Chapter 0
BAD_BITS = sum(1 << b for b in [0, 1, 2, 3, 4, 5, 6, 7, 12])   # pixels given zero weight (Chapter 0)


def prepare(sp, rows=slice(None)):
    """flux and err (float64) of the spectra file sp; err = inf where there are no data, a BAD_BITS flag, or
    outside CHIPS (the detector edges, where only some spectra have data)."""
    flux = sp['flux'][rows].astype(float)
    err = sp['err'][rows].astype(float)
    on_chip = np.any([(wave > lo) & (wave < hi) for lo, hi in CHIPS], axis=0)   # pixels at the detector edges and
    bad = ((sp['mask'][rows] & BAD_BITS) > 0) | ~np.isfinite(flux) | ~np.isfinite(err) | (err <= 0) | ~on_chip  # gaps: no weight
    return np.where(bad, 0.0, flux), np.where(bad, np.inf, err)


def cheb_normalise(flux, err, cont, order=2):
    """Divide by a Chebyshev polynomial of the given order fitted, per detector, to the continuum pixels `cont`
    (boolean, NPIX) with weights 1/err^2 (N15 sect. 2.3 and 5.3). Returns normalised flux and error;
    pixels outside the detectors get err = inf."""
    fn, en = np.zeros_like(flux), np.full_like(err, np.inf)
    for lo, hi in CHIPS:
        c = (wave > lo) & (wave < hi)
        t = (wave[c] - (lo + hi) / 2) / ((hi - lo) / 2)          # wavelength mapped to [-1, 1]
        B = np.polynomial.chebyshev.chebvander(t, order)         # (pixels, order+1) basis
        use = cont[c]
        w = 1 / err[:, c][:, use]**2                             # (N, continuum pixels); 0 for flagged pixels
        # one small weighted least-squares problem per star: (B^T W B) a = B^T W f
        BtWB = np.einsum('pi,np,pj->nij', B[use], w, B[use])
        BtWf = np.einsum('pi,np->ni', B[use], w * flux[:, c][:, use])
        a = np.linalg.solve(BtWB, BtWf[:, :, None])[:, :, 0]     # (N, order+1); b as a stack of column vectors
        cont_fit = a @ B.T                                       # (N, pixels of this detector)
        fn[:, c] = flux[:, c] / cont_fit
        en[:, c] = err[:, c] / cont_fit
    return fn, en
# --- The Cannon (as written in Chapter 2) -----------------------------------------------------------------
from scipy.optimize import least_squares


def vectorize(x):
    """Quadratic label vector v(x) = [1, x_k, x_k x_l (k <= l)] for scaled labels x of shape (N, K): (N, D)."""
    x = np.atleast_2d(x)
    k, l = np.triu_indices(x.shape[1])
    return np.hstack([np.ones((len(x), 1)), x, x[:, k] * x[:, l]])


def dvectorize(x):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    K = len(x)
    k, l = np.triu_indices(K)
    J = np.zeros((1 + K + len(k), K))
    J[1:1 + K] = np.eye(K)
    for i, (a, b) in enumerate(zip(k, l)):
        J[1 + K + i, a] += x[b]
        J[1 + K + i, b] += x[a]
    return J


def fit_theta(V, flux, var):
    """Weighted least squares at every pixel: theta (P, D) and its covariance (P, D, D); weights 1/var (N, P).
    Zero-weight entries (var = inf) drop out of the sums."""
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)                       # NaN flux at zero weight must not poison the sums
    N, Dv = V.shape
    A = ((V[:, :, None] * V[:, None, :]).reshape(N, Dv * Dv).T @ w).T.reshape(-1, Dv, Dv)
    b = (V.T @ (w * flux)).T
    cov = np.linalg.pinv(A)                                 # pinv: pixels without data give theta = 0, not an error
    return np.einsum('pde,pe->pd', cov, b), cov


S_GRID = np.r_[0, np.logspace(-4, 0, 41)]                  # 0, then 1e-4 ... 1 in steps of 0.1 dex


def fit_scatter(resid, err2):
    """Maximum-likelihood scatter s per pixel for fixed residuals: best value on the grid S_GRID, refined by a
    parabola in log s (Chapter 2). Zero-weight entries (err2 = inf) are ignored."""
    use = np.isfinite(err2) & np.isfinite(resid)
    r2 = np.where(use, resid**2, 0.0)
    e2 = np.where(use, err2, 1.0)
    nll = np.array([np.sum(use * (r2 / (e2 + s**2) + np.log(e2 + s**2)), axis=0) for s in S_GRID])
    best = np.argmin(nll, axis=0)
    j = np.clip(best, 2, len(S_GRID) - 2)
    p = np.arange(nll.shape[1])
    a, b, c = nll[j - 1, p], nll[j, p], nll[j + 1, p]
    shift = np.clip(0.5 * (a - c) / np.where(a - 2 * b + c > 0, a - 2 * b + c, np.inf), -1, 1)   # stay near the grid optimum
    return np.where(best == 0, 0.0, S_GRID[j] * np.exp(shift * np.log(S_GRID[2] / S_GRID[1])))


def train(x, flux, err, n_iter=3):
    """Training step. x: scaled labels (N, K); flux, err: (N, P) (err = inf for pixels to ignore).
    Returns theta (P, D), its covariance (P, D, D) and the scatter s (P,)."""
    V, err2 = vectorize(x), err**2
    s = np.zeros(flux.shape[1])
    for _ in range(n_iter):
        theta, cov = fit_theta(V, flux, err2 + s**2)
        s = fit_scatter(flux - V @ theta.T, err2)
    theta, cov = fit_theta(V, flux, err2 + s**2)
    return theta, cov, s


def test(flux, err, theta, s, x0=None):
    """Test step for one spectrum: scaled labels (K,), covariance (K, K), chi^2, number of pixels used, converged?"""
    K = int(round((np.sqrt(8 * theta.shape[1] + 1) - 3) / 2))   # invert D = 1 + K + K(K+1)/2
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize(x)[0]) * w
    jac = lambda x: -(th @ dvectorize(x)) * w[:, None]
    r = least_squares(res, np.zeros(K) if x0 is None else x0, jac=jac, method='lm')
    # covariance from the Jacobian at the optimum (pinv: no crash if a label is unconstrained); r.success: converged
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def train_clean(x, flux, err, cut=3.0):
    """Train, drop the training stars whose mean squared residual per pixel (in units of sigma^2 + s^2) exceeds
    `cut`, and train again (Chapter 4). Returns theta, its covariance, s, and the boolean mask of stars kept."""
    theta, cov, s = train(x, flux, err)
    z2 = (flux - vectorize(x) @ theta.T)**2 / (err**2 + s**2)
    z2 = np.where(np.isfinite(err) & (theta[:, 0] != 0), z2, np.nan)
    keep = np.nanmean(z2, axis=1) < cut
    theta, cov, s = train(x[keep], flux[keep], err[keep])
    return theta, cov, s, keep


def scatter_stats(d):
    """Summary of differences d (e.g. Cannon - reference): mean (bias), standard deviation, root-mean-square error,
    robust standard deviation 1.4826 x median absolute deviation (= std for a Gaussian, insensitive to outliers),
    and the fraction of points more than 3 robust std from the median (the tails)."""
    d = np.asarray(d)[np.isfinite(d)]
    med = np.median(d)
    rob = 1.4826 * np.median(np.abs(d - med))
    return {'bias': d.mean(), 'std': d.std(), 'rmse': np.sqrt(np.mean(d**2)), 'robust': rob,
            'tail': np.mean(np.abs(d - med) > 3 * rob)}
import time
import logging
import io
import contextlib
import subprocess
import sys
try:
    import thecannon as tc
except ImportError:                                              # e.g. in Colab
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'annieslasso==0.2.96', 'six'], check=True)
    import thecannon as tc

logging.getLogger('thecannon').setLevel(logging.ERROR)   # AnniesLasso logs every spectrum at DEBUG level
if not hasattr(np.linalg, 'linalg'):
    np.linalg.linalg = np.linalg   # AnniesLasso 0.2.96 catches np.linalg.linalg.LinAlgError, removed in numpy 2
_fit_pixel = tc.fitting.fit_pixel_fixed_scatter


def fit_pixel_scalar_s2(*args, **kwargs):
    """AnniesLasso 0.2.96 returns s^2 as a length-1 array, which numpy 2 will not store in a scalar slot."""
    theta, s2, meta = _fit_pixel(*args, **kwargs)
    return theta, float(np.squeeze(s2)), meta


tc.fitting.fit_pixel_fixed_scatter = fit_pixel_scalar_s2
labels = load('labels')
cont = load('continuum_pixels')['continuum'].to_numpy()
NAMES, TEX, UNITS = ['teff', 'logg', 'fe_h'], [r'$T_{\rm eff}$', r'$\log g$', '[Fe/H]'], ['K', 'dex', 'dex']
data = {}
for nm in ['train', 'test']:
    lab = labels[labels['sample'] == nm].reset_index(drop=True)
    f, e = cheb_normalise(*prepare(load(f'spectra_{nm}')), cont)
    data[nm] = (lab[NAMES].to_numpy(), f, e)
L, flux, err = data['train']
Lt, ft, et = data['test']
pivot = np.median(L, axis=0)
scale = (np.percentile(L, 97.5, axis=0) - np.percentile(L, 2.5, axis=0)) / 2
x = (L - pivot) / scale


def to_ivar(f, e):
    """AnniesLasso takes inverse variances; pixels we ignore (err = inf) get ivar = 0 and a finite dummy flux."""
    ivar = np.where(np.isfinite(e), 1 / e**2, 0.0)
    return np.where(ivar > 0, f, 1.0), ivar


print(f'{len(L)} training stars, {len(Lt)} test stars, {flux.shape[1]} pixels, AnniesLasso {tc.__version__}')

## A.1 Training both codes

AnniesLasso fits each pixel with an iterative optimiser, ours solves all pixels at once by linear algebra; both
take about ten seconds. Running AnniesLasso 0.2.96 with numpy 2 needs three small workarounds, all in the cells
above and below: a keyword for the label scale that it rejects, an exception class that numpy 2 removed, and a
scatter returned as an array of length one. None of them changes a number.

In [ ]:
fa, ia = to_ivar(flux, err)
vec = tc.vectorizer.PolynomialVectorizer(NAMES, 2)
al = tc.CannonModel(L, fa, ia, vec, regularization=0.0)
# use our scale (half AnniesLasso's); the keyword __scale_labels_function meant for this is rejected by 0.2.96,
# so set the scale and rebuild the design matrix as CannonModel.__init__ does
al._scales = scale
al._design_matrix = vec((L - al._fiducials) / al._scales).T
t0 = time.time()
with contextlib.redirect_stdout(io.StringIO()):     # AnniesLasso draws a progress bar per pixel
    th_al, s2_al, _ = al.train()
t_al = time.time() - t0
s_al = np.sqrt(s2_al)
t0 = time.time()
th_us, _, s_us = train(x, flux, err)
t_us = time.time() - t0
print(f'training time: AnniesLasso {t_al:.0f} s, ours {t_us:.1f} s')
print('same label vector:', vec.human_readable_label_vector)
assert np.allclose(al._fiducials, pivot) and np.allclose(al._scales, scale)

## A.2 Run the same way, the two codes must agree

**Training step.** With the scatter set to zero our `fit_theta` (Chapter 2) solves the same weighted least-squares
problem that AnniesLasso minimises. Compare the two spectral models at the labels of the training stars, in units
of the flux errors. The pixels that AnniesLasso skips (too little weight: it sets $\boldsymbol\theta = (1, 0, \dots)$
and $s^2 = \infty$) are the pixels that our `prepare` gave zero weight; they are left out.

**Test step.** Give our `test` the coefficients and scatter of AnniesLasso; both must then find the same labels,
unless one of the optimisers stops before the minimum. Where they differ, the $\chi^2$ of the two solutions under
the same model says which one stopped early.

In [ ]:
V = vectorize(x)
th0, _ = fit_theta(V, flux, err**2)                         # our weighted least squares with s = 0
use = np.isfinite(s2_al)
print(f'pixels used: AnniesLasso {use.sum()}, ours {(th_us[:, 0] != 0).sum()}')
dmod = (V @ (th0[use] - th_al[use]).T) / np.where(np.isfinite(err[:, use]), err[:, use], np.inf)
print(f'|model difference| / sigma: median {np.median(np.abs(dmod)):.1e}, 99.9th percentile '
      f'{np.percentile(np.abs(dmod), 99.9):.1e}, max {np.abs(dmod).max():.1e}')

fta, ita = to_ivar(ft, et)
t0 = time.time()
with contextlib.redirect_stdout(io.StringIO()):
    lab_al, cov_al, meta_al = al.test(fta, ita)
t_test_al = time.time() - t0
s_for_us = np.where(use, s_al, 0.0)
th_for_us = np.where(use[:, None], th_al, 0.0)                # theta_0 = 0: our test skips these pixels
t0 = time.time()
out = [test(ft[n], et[n], th_for_us, s_for_us) for n in range(len(ft))]
t_test_us = time.time() - t0
lab_same = np.array([o[0] for o in out]) * scale + pivot
d_same = lab_same - lab_al
print(f'test step time: AnniesLasso {t_test_al:.0f} s, ours {t_test_us:.0f} s')
sig_same = np.array([np.sqrt(np.diag(o[1])) for o in out]) * scale
z_same = np.abs(d_same / sig_same).max(axis=1)                  # largest difference over the 3 labels, in sigma


def chi2(n, lab):
    """chi^2 of spectrum n at physical labels lab under the AnniesLasso model (the function minimised by both)."""
    ok = np.isfinite(ft[n]) & np.isfinite(et[n]) & use
    r = (ft[n] - th_for_us @ vectorize((lab - pivot) / scale)[0]) / np.sqrt(et[n]**2 + s_for_us**2)
    return np.sum(r[ok]**2)


far = np.flatnonzero(z_same > 0.1)
dchi = np.array([chi2(n, lab_al[n]) - chi2(n, lab_same[n]) for n in far])
print(f'|ours - AnniesLasso| / formal error, largest of the 3 labels: median {np.median(z_same):.4f}, '
      f'99th percentile {np.percentile(z_same, 99):.3f}')
print(f'{len(far)} stars differ by more than 0.1 sigma; chi^2(AnniesLasso) - chi^2(ours) for them: '
      f'min {dchi.min():.2f}, median {np.median(dchi):.1f}, max {dchi.max():.1f}')

**Checks.** The spectral models agree to a millionth of the noise. The labels of 99% of the stars agree to a few
hundredths of their formal errors; for the two that differ by more than 0.1 of their errors, AnniesLasso's solution
has the higher (or the same) $\chi^2$: its optimiser (`scipy.optimize.leastsq`) stopped before the minimum. (AnniesLasso returns `NaN`
labels when the optimiser does not move from the starting point; there are none.)

In [ ]:
n_nan = np.isnan(lab_al).any(axis=1).sum()
print('AnniesLasso NaN labels:', n_nan)
assert np.abs(dmod).max() < 1e-3, 'the two least-squares solutions differ'
assert n_nan == 0
assert np.percentile(z_same, 99) < 0.05, 'the test steps disagree'
assert np.all(dchi > -1e-2), 'AnniesLasso found a lower chi^2 than our test step'

## A.3 Each code as it is meant to be run

Now compare AnniesLasso ($\boldsymbol\theta$ fitted without scatter, then $s$ from "median $\chi^2 = 1$") with our
`train` ($\boldsymbol\theta$ and $s$ iterated, $s$ by maximum likelihood). The labels of the 1500 test stars from
both are compared with each other and with ASPCAP. To see which of the two differences matters, also label the
test stars with our $\boldsymbol\theta$ and AnniesLasso's $s$, and the other way round.

In [ ]:
def label_all(theta, s):
    """Our test step on all test spectra: physical labels and formal errors."""
    out = [test(ft[n], et[n], theta, s) for n in range(len(ft))]
    assert all(o[4] for o in out)
    return np.array([o[0] for o in out]) * scale + pivot, np.array([np.sqrt(np.diag(o[1])) for o in out]) * scale


lab_us, sig_us = label_all(th_us, s_us)
lab_mix = {'our theta, AL s': label_all(th_us, s_for_us)[0], 'AL theta, our s': label_all(th_for_us, s_us)[0]}
d_codes = lab_us - lab_al
both = use & (th_us[:, 0] != 0)
print(f'scatter, median over pixels: ours {np.median(s_us[both]):.4f}, AnniesLasso {np.median(s_al[both]):.4f}; '
      f'median ratio AnniesLasso / ours (s > 0.002) = {np.median((s_al / s_us)[both & (s_us > 0.002)]):.2f}')
r = flux[:, use] - V @ th_al[use].T
z2 = np.where(np.isfinite(err[:, use]), r**2 / (err[:, use]**2 + s2_al[use]), np.nan)
print(f'at the AnniesLasso scatter, training stars: median chi^2 per pixel {np.median(np.nanmedian(z2, axis=0)):.2f}, '
      f'mean chi^2 per pixel {np.median(np.nanmean(z2, axis=0)):.2f} (medians over pixels)')
rob = lambda d: [scatter_stats(d[:, k])['robust'] for k in range(3)]
print(f"{'robust std of':34s}" + ''.join(f'{n:>10s}' for n in NAMES))
for name, d in [('ours - ASPCAP', lab_us - Lt), ('AnniesLasso - ASPCAP', lab_al - Lt), ('ours - AnniesLasso', d_codes)] + \
               [(f'ours - ({k})', lab_us - v) for k, v in lab_mix.items()]:
    print(f'{name:34s}' + ''.join(f'{v:10.3f}' for v in rob(d)))
print(f"{'ours - AnniesLasso, in formal errors':34s}" + ''.join(f'{v:10.2f}' for v in rob(d_codes / sig_us)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
ok = both & (s_us > 0) & (s_al > 0)
ax[0].loglog(s_us[ok], s_al[ok], ',', color='k', alpha=0.5)
ax[0].plot([1e-4, 1], [1e-4, 1], 'r--', lw=1)
ax[0].set(xlim=(1e-4, 0.3), ylim=(1e-4, 0.3), xlabel=r'$s_\lambda$, ours (maximum likelihood)',
          ylabel=r'$s_\lambda$, AnniesLasso (median $\chi^2=1$)', title='(a) scatter at each pixel')
for k, c in enumerate(['C0', 'C1', 'C2']):
    ax[1].hist(d_codes[:, k] / sig_us[:, k], bins=np.linspace(-5, 5, 81), histtype='step', color=c, lw=1.5,
               label=TEX[k])
ax[1].set(xlabel=r'(ours $-$ AnniesLasso) / formal error', ylabel='test stars', title='(b) the two codes')
ax[1].legend()
k = 2
for lab, c, nm in [(lab_us, 'C3', 'ours'), (lab_al, 'C0', 'AnniesLasso')]:
    ax[2].hist(lab[:, k] - Lt[:, k], bins=np.linspace(-0.3, 0.3, 61), histtype='step', color=c, lw=1.5, label=nm)
ax[2].set(xlabel='[Fe/H] $-$ [Fe/H]$_{\\rm ASPCAP}$ [dex]', ylabel='test stars', title='(c) against ASPCAP')
ax[2].legend()
plt.tight_layout()
plt.show()

**Figure A.1.** Our Cannon and AnniesLasso 0.2.96 trained on the same 1703 APOGEE giants and applied to the same
1500 test stars (S/N 50–100). **(a)** The scatter $s_\lambda$ at every pixel: AnniesLasso's rule (median
$\chi^2 = 1$ over the training stars) against our maximum-likelihood value; the dashed line is equality.
AnniesLasso's scatter is smaller at every pixel, by a factor of about two. **(b)** The difference between the
labels from the two codes, in units of the formal error of our labels (which Chapter 6 showed to be 2–4 times
smaller than the real errors); the differences are systematic, they depend on where the star lies in label space.
**(c)** [Fe/H] from each code minus ASPCAP: the two distributions nearly coincide.

Why the scatters differ: the residuals of the training stars at a pixel have heavy tails (a few stars that the
three labels do not describe). The median ignores them; the likelihood does not, and gives a larger $s$. At
AnniesLasso's scatter the median $\chi^2$ is 1 by construction, but the mean $\chi^2$ is about 3 (printed above).
For Gaussian residuals the rule "median $\chi^2 = 1$" would instead *overestimate* $s$: the median of $\chi^2$
with one degree of freedom is 0.45, not 1.

**Checks.** The two codes agree with each other much better than either agrees with ASPCAP (the difference is
about a fifth of the scatter about ASPCAP). Most of the difference comes from $\boldsymbol\theta$, not from $s$:
swapping only the scatter moves the labels half as much as swapping only the coefficients. On these data ours is
slightly closer to ASPCAP in every label (by 4–10%): weighting the training stars by $1/(\sigma^2+s^2)$ rather
than $1/\sigma^2$ stops the few highest-S/N stars from dominating the fit at pixels where the labels do not
describe the spectra to within their noise.

In [ ]:
ru, ra, rc = map(np.array, (rob(lab_us - Lt), rob(lab_al - Lt), rob(d_codes)))
assert np.all(s_al[both] <= s_us[both])
assert np.all(rc < 0.3 * ru), 'the two codes differ by more than a third of the scatter about ASPCAP'
assert np.all(ru <= ra), 'ours is not closer to ASPCAP'
assert np.all(np.array(rob(lab_us - lab_mix['our theta, AL s'])) < np.array(rob(lab_us - lab_mix['AL theta, our s'])))

## Summary

- Run the same way, our code and AnniesLasso 0.2.96 agree: the spectral models to $10^{-6}$ of the noise, the labels
  of 99% of the test stars to a few hundredths of their formal errors. Where the labels differ more (2 of the 1500
  stars), AnniesLasso's optimiser stopped early, at a higher $\chi^2$ (or at the same $\chi^2$ in a flat valley). The implementation of Chapter 2 has no
  bug that this comparison can see.
- Run as designed, the codes differ: AnniesLasso fits $\boldsymbol\theta$ without the scatter and sets $s$ by
  "median $\chi^2 = 1$", we iterate both with $s$ by maximum likelihood. Its scatter is half ours; its labels differ
  from ours by about a fifth of the scatter about ASPCAP, mostly because of the weights in the fit of
  $\boldsymbol\theta$, and are slightly further from ASPCAP.
- Running AnniesLasso 0.2.96 with numpy 2 needed three small workarounds (no change to any number).

## References

- Casey, A. R., Hogg, D. W., Ness, M., et al. 2016, *The Cannon 2: A data-driven model of stellar spectra for
  detailed chemical abundance analyses*, [arXiv:1603.03040](https://ui.adsabs.harvard.edu/abs/2016arXiv160303040C)
  — the code: [AnniesLasso on GitHub](https://github.com/andycasey/AnniesLasso),
  [annieslasso on PyPI](https://pypi.org/project/annieslasso/), [documentation](https://annieslasso.readthedocs.io).
- Ness, M., Hogg, D. W., Rix, H.-W., Ho, A. Y. Q., & Zasowski, G. 2015, *The Cannon: a data-driven approach to
  stellar label determination*, [ApJ 808, 16](https://ui.adsabs.harvard.edu/abs/2015ApJ...808...16N).